# 01 Check Data

This notebook loads the raw Phase 1 datasets, cleans them into a consistent schema, and builds `master.csv`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data')
CLEAN_DIR = DATA_DIR / 'cleaned'
CLEAN_DIR.mkdir(exist_ok=True)

STATE_MAP = {
    'Andaman & Nicobar Islands': 'Andaman And Nicobar Islands',
    'Andaman and Nicobar Island': 'Andaman And Nicobar Islands',
    'Jammu & Kashmir': 'Jammu And Kashmir',
    'Jammu and Kashmir': 'Jammu And Kashmir',
    'Dadra and Nagar Haveli & Daman and Diu': 'The Dadra And Nagar Haveli And Daman And Diu',
    'Dadra & Nagar Haveli and Daman & Diu': 'The Dadra And Nagar Haveli And Daman And Diu',
    'Dadra and Nagar Haveli': 'The Dadra And Nagar Haveli And Daman And Diu',
    'NCT of Delhi': 'Delhi',
    'Delhi': 'Delhi',
    'Orissa': 'Odisha',
}

def normalize_state(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().replace(STATE_MAP)

In [2]:
files = {
    'Food Prices': DATA_DIR / 'food_prices.csv',
    'Rainfall': DATA_DIR / 'rainfall.csv',
    'Nutrition': DATA_DIR / 'nutrition.csv',
    'Crop Old': DATA_DIR / 'crop_old.csv',
    'Crop New': DATA_DIR / 'crop_new.csv',
}

for name, path in files.items():
    df = pd.read_csv(path)
    print(f"\\n{'=' * 40}")
    print(name)
    print(path.name)
    print(f'Shape: {df.shape}')
    print(df.head(2))

\n========================================
Food Prices
food_prices.csv
Shape: (2117473, 6)
   id        date      state_name  state_code commodity  price
0   0  2015-01-01  Andhra Pradesh          28      Rice   26.0
1   1  2015-01-01           Assam          18      Rice   24.0
\n========================================
Rainfall
rainfall.csv
Shape: (204876, 8)
   id        date  state_code   state_name  actual       rfs  normal  \
0   0  2009-01-01           5  Uttarakhand     0.0  0.003906    2.19   
1   1  2009-01-01          18        Assam     0.0  0.000000    0.52   

   deviation  
0     -100.0  
1     -100.0  
\n========================================
Nutrition
nutrition.csv
Shape: (110, 137)
  States/UTs   Area  Number of Households surveyed  \
0      India  Urban                         160138   
1      India  Rural                         476561   

   Number of Women age 15-49 years interviewed  \
0                                       179535   
1                         

In [3]:
# Food prices\n
food_raw = pd.read_csv(DATA_DIR / 'food_prices.csv')

target_commodities = ['Rice', 'Wheat', 'Tur/Arhar Dal', 'Soya Oil (Packed)']
food = food_raw[food_raw['commodity'].isin(target_commodities)].copy()
food['state'] = normalize_state(food['state_name'])
food['date'] = pd.to_datetime(food['date'])
food['price'] = pd.to_numeric(food['price'], errors='coerce')

food_wide = food.pivot_table(
    index=['state', 'date'],
    columns='commodity',
    values='price',
    aggfunc='mean'
).reset_index()

food_wide.columns.name = None
food_wide = food_wide.rename(columns={
    'Rice': 'rice_price',
    'Wheat': 'wheat_price',
    'Tur/Arhar Dal': 'dal_price',
    'Soya Oil (Packed)': 'oil_price',
})

food_wide['date'] = food_wide['date'].dt.to_period('M').dt.to_timestamp()
food_cleaned = food_wide.groupby(['state', 'date'], as_index=False).mean(numeric_only=True)
food_cleaned.to_csv(CLEAN_DIR / 'food_prices_cleaned.csv', index=False)

print(food_cleaned.head(3))
print(food_cleaned.isnull().sum())

                         state       date  rice_price  oil_price   dal_price  \
0  Andaman And Nicobar Islands 2015-01-01       40.03        NaN  114.849091   
1  Andaman And Nicobar Islands 2015-02-01       36.00        NaN   90.750000   
2  Andaman And Nicobar Islands 2015-03-01       36.00        NaN   96.470588   

   wheat_price  
0       34.375  
1       27.000  
2       27.000  
state            0
date             0
rice_price       0
oil_price      416
dal_price       85
wheat_price    354
dtype: int64


In [4]:
# Rainfall\n
rain = pd.read_csv(DATA_DIR / 'rainfall.csv')
rain = rain[['date', 'state_name', 'actual']].copy()
rain = rain.rename(columns={'state_name': 'state', 'actual': 'rainfall_mm'})
rain['state'] = normalize_state(rain['state'])
rain['date'] = pd.to_datetime(rain['date'])
rain['rainfall_mm'] = pd.to_numeric(rain['rainfall_mm'], errors='coerce')
rain.to_csv(CLEAN_DIR / 'rainfall_cleaned.csv', index=False)

print(rain.head(3))
print(rain.isnull().sum())

        date        state  rainfall_mm
0 2009-01-01  Uttarakhand          0.0
1 2009-01-01        Assam          0.0
2 2009-01-01      Tripura          0.0
date               0
state              0
rainfall_mm    17162
dtype: int64


In [5]:
# Nutrition\n
nutrition = pd.read_csv(DATA_DIR / 'nutrition.csv')
nutrition = nutrition[[
    'States/UTs',
    'Area',
    'Children under 5 years who are stunted (height-for-age)18 (%)',
    'Children under 5 years who are underweight (weight-for-age)18 (%)',
    'Children age 6-59 months who are anaemic (<11.0 g/dl)22 (%)',
]].copy()

nutrition = nutrition[nutrition['Area'].astype(str).str.strip().eq('Total')].copy()
nutrition = nutrition[~nutrition['States/UTs'].astype(str).str.strip().eq('India')].copy()
nutrition.columns = ['state', 'area', 'stunting_rate', 'underweight_rate', 'anaemia_rate']
nutrition['state'] = normalize_state(nutrition['state'])

for col in ['stunting_rate', 'underweight_rate', 'anaemia_rate']:
    nutrition[col] = nutrition[col].astype(str).str.replace(r'[^0-9.]', '', regex=True)
    nutrition[col] = pd.to_numeric(nutrition[col], errors='coerce')

nutrition = nutrition.drop(columns=['area'])
nutrition = nutrition.dropna(subset=['state']).drop_duplicates(subset=['state']).reset_index(drop=True)
nutrition.to_csv(CLEAN_DIR / 'nutrition_cleaned.csv', index=False)

print(nutrition.head(5))
print('Rows:', len(nutrition), 'Unique states:', nutrition['state'].nunique())

                         state  stunting_rate  underweight_rate  anaemia_rate
0  Andaman And Nicobar Islands           22.5              23.7          40.0
1               Andhra Pradesh           31.2              29.6          63.2
2            Arunachal Pradesh           28.0              15.4          56.6
3                        Assam           35.3              32.8          68.4
4                        Bihar           42.9              41.0          69.4
Rows: 35 Unique states: 35


In [6]:
# Crop yield / productivity\n
CROP_OLD_MAP = {
    'Rice': 'Rice',
    'Wheat': 'Wheat',
    'Maize': 'Maize',
    'Bajra': 'Bajra',
    'Gram': 'Gram',
    'Masoor': 'Lentil',
    'Rapeseed &Mustard': 'Rapeseed and Mustard',
    'Sugarcane': 'Sugarcane',
}

CROP_NEW_MAP = {
    'Productivity of Rice': 'Rice',
    'Productivity of Wheat': 'Wheat',
    'Productivity of Maize': 'Maize',
    'Productivity of Bajra': 'Bajra',
    'Productivity of Gram': 'Gram',
    'Productivity of Lentil': 'Lentil',
    'Productivity of Rapeseed and Mustard': 'Rapeseed and Mustard',
    'Productivity of Sugarcane': 'Sugarcane',
}

COMMON_CROPS = sorted(set(CROP_OLD_MAP.values()) & set(CROP_NEW_MAP.values()))

crop_old = pd.read_csv(DATA_DIR / 'crop_old.csv')
crop_old.columns = [c.strip().lower() for c in crop_old.columns]
crop_old['state'] = normalize_state(crop_old['state'])
crop_old['crop'] = crop_old['crop'].astype(str).str.strip().replace(CROP_OLD_MAP)
crop_old = crop_old[crop_old['crop'].isin(COMMON_CROPS)].copy()

for col in ['area', 'production']:
    crop_old[col] = pd.to_numeric(crop_old[col], errors='coerce')

crop_old = crop_old.groupby(['state', 'crop', 'crop_year'], as_index=False).agg(
    area=('area', 'sum'),
    production=('production', 'sum')
)
crop_old['year'] = crop_old['crop_year'].astype(int)
crop_old['yield'] = np.where(crop_old['area'] > 0, crop_old['production'] / crop_old['area'], np.nan)
crop_old = crop_old[['state', 'crop', 'year', 'yield', 'area', 'production']]

crop_new = pd.read_csv(DATA_DIR / 'crop_new.csv')
crop_new.columns = ['sl_no', 'state', 'category', '2021', '2022', '2023']
crop_new['state'] = normalize_state(crop_new['state'])
crop_new['crop'] = crop_new['category'].replace(CROP_NEW_MAP)
crop_new = crop_new[crop_new['crop'].isin(COMMON_CROPS)].copy()
crop_new = crop_new.melt(
    id_vars=['state', 'crop'],
    value_vars=['2021', '2022', '2023'],
    var_name='year',
    value_name='yield'
)
crop_new['year'] = crop_new['year'].astype(int)
crop_new['yield'] = pd.to_numeric(crop_new['yield'], errors='coerce')
crop_new['area'] = np.nan
crop_new['production'] = np.nan
crop_new = crop_new[['state', 'crop', 'year', 'yield', 'area', 'production']]

crop_combined = pd.concat([crop_old, crop_new], ignore_index=True, sort=False)
crop_combined = crop_combined.sort_values(['state', 'crop', 'year']).reset_index(drop=True)
crop_combined.to_csv(CLEAN_DIR / 'crop_yield_by_crop_cleaned.csv', index=False)

crop_summary = crop_combined.groupby(['state', 'year'], as_index=False).apply(
    lambda g: pd.Series({
        'yield_ton_per_hectare': (
            (g['yield'] * g['area']).sum() / g['area'].sum()
            if g['area'].sum() > 0 else g['yield'].mean()
        ),
        'production_tonnes': g['production'].sum(min_count=1)
    })
).reset_index(drop=True)
crop_summary.to_csv(CLEAN_DIR / 'crop_yield_cleaned.csv', index=False)

print('Common crops:', COMMON_CROPS)
print(crop_summary.head(5))
print('Year range:', crop_summary['year'].min(), 'to', crop_summary['year'].max())

Common crops: ['Bajra', 'Gram', 'Lentil', 'Maize', 'Rapeseed and Mustard', 'Rice', 'Sugarcane', 'Wheat']
                         state  year  yield_ton_per_hectare  production_tonnes
0  Andaman And Nicobar Islands  2000               3.258822            35922.0
1  Andaman And Nicobar Islands  2001               3.006476            29713.0
2  Andaman And Nicobar Islands  2002               4.046893            45912.0
3  Andaman And Nicobar Islands  2003               3.114770            33731.0
4  Andaman And Nicobar Islands  2004               2.826516            30753.0
Year range: 1997 to 2023


In [7]:
# Build master.csv\n
food = pd.read_csv(CLEAN_DIR / 'food_prices_cleaned.csv')
rain = pd.read_csv(CLEAN_DIR / 'rainfall_cleaned.csv')
nutrition = pd.read_csv(CLEAN_DIR / 'nutrition_cleaned.csv')
crop = pd.read_csv(CLEAN_DIR / 'crop_yield_cleaned.csv')

food['date'] = pd.to_datetime(food['date'])
rain['date'] = pd.to_datetime(rain['date'])
food['year'] = food['date'].dt.year

master = food.merge(rain, on=['state', 'date'], how='left')
master = master.merge(crop, on=['state', 'year'], how='left')
master = master.merge(nutrition, on='state', how='left')
master.to_csv(DATA_DIR / 'master.csv', index=False)

print(f'Master shape: {master.shape}')
print(master.head(3))
print('\\nNull counts:')
print(master.isnull().sum())

states_food = set(food['state'].dropna().unique())
states_rain = set(rain['state'].dropna().unique())
states_nutrition = set(nutrition['state'].dropna().unique())
states_crop = set(crop['state'].dropna().unique())

print('\\nIn food but NOT in rainfall:')
print(sorted(states_food - states_rain))
print('\\nIn food but NOT in nutrition:')
print(sorted(states_food - states_nutrition))
print('\\nIn food but NOT in crop:')
print(sorted(states_food - states_crop))

Master shape: (3741, 13)
                         state       date  rice_price  oil_price   dal_price  \
0  Andaman And Nicobar Islands 2015-01-01       40.03        NaN  114.849091   
1  Andaman And Nicobar Islands 2015-02-01       36.00        NaN   90.750000   
2  Andaman And Nicobar Islands 2015-03-01       36.00        NaN   96.470588   

   wheat_price  year  rainfall_mm  yield_ton_per_hectare  production_tonnes  \
0       34.375  2015          NaN               2.586068            16383.0   
1       27.000  2015          NaN               2.586068            16383.0   
2       27.000  2015          NaN               2.586068            16383.0   

   stunting_rate  underweight_rate  anaemia_rate  
0           22.5              23.7          40.0  
1           22.5              23.7          40.0  
2           22.5              23.7          40.0  
\nNull counts:
state                       0
date                        0
rice_price                  0
oil_price                 41